# Blink-to-Morse-Code Decoder

A computer-vision prototype that interprets eye-blink durations as Morse code using **Python**, **OpenCV**, and **MediaPipe Face Landmarker**.

This notebook contains two implementations:
1. **Live webcam decoder** — decode blink input in real time.
2. **Pre-recorded video decoder** — process a video file and save the decoded text and event log.

> **Note:** This is an experimental prototype. Blink timing thresholds may need calibration for different users and camera conditions.


## 1. Setup

Run the following cell once in your notebook environment. A working webcam is required for the live decoder. The prerecorded-video section requires a video file that you provide.


In [ ]:
%pip install mediapipe opencv-python numpy


## 2. Live webcam decoder

Run the next cell to start the webcam decoder. Click the camera window to use the keyboard controls:

- `b` — delete the last character
- `space` — insert a word space
- `r` — reset the decoded message
- `q` — quit

The MediaPipe face-landmark model is downloaded automatically on first run if it is not already present.


In [ ]:
"""
Blink-to-Morse-Code Decoder
---------------------------
MediaPipe Tasks API version

DOT   = eyes closed for 0.5 to less than 3 seconds
DASH  = eyes closed for 3 seconds or more
LETTER GAP = eyes open for 3 seconds
WORD SPACE = eyes open for 6 seconds

Keyboard controls (click the camera window first):
b = backspace
r = reset
q = quit
"""

import cv2
import time
import numpy as np
import mediapipe as mp
import urllib.request
import os

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ============================================================
# 1. MORSE CODE TABLE
# ============================================================

MORSE_TO_CHAR = {
    '.-': 'A', '-...': 'B', '-.-.': 'C', '-..': 'D', '.': 'E',
    '..-.': 'F', '--.': 'G', '....': 'H', '..': 'I', '.---': 'J',
    '-.-': 'K', '.-..': 'L', '--': 'M', '-.': 'N', '---': 'O',
    '.--.': 'P', '--.-': 'Q', '.-.': 'R', '...': 'S', '-': 'T',
    '..-': 'U', '...-': 'V', '.--': 'W', '-..-': 'X', '-.--': 'Y',
    '--..': 'Z',
    '-----': '0', '.----': '1', '..---': '2', '...--': '3',
    '....-': '4', '.....': '5', '-....': '6', '--...': '7',
    '---..': '8', '----.': '9'
}


# ============================================================
# 2. FACE LANDMARK MODEL
# ============================================================

MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/face_landmarker/face_landmarker/float16/1/"
    "face_landmarker.task"
)
MODEL_PATH = "face_landmarker.task"


def download_model():
    if os.path.exists(MODEL_PATH):
        print("Face Landmarker model already exists.")
        return

    print("Downloading Face Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded successfully.")


# ============================================================
# 3. TIMESTAMP
# ============================================================

_frame_timestamp_ms = 0


def next_timestamp():
    global _frame_timestamp_ms
    _frame_timestamp_ms += 1
    return _frame_timestamp_ms


# ============================================================
# 4. EYE LANDMARKS
# ============================================================

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]


# ============================================================
# 5. EUCLIDEAN DISTANCE
# ============================================================

def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


# ============================================================
# 6. EYE ASPECT RATIO
# ============================================================

def eye_aspect_ratio(landmarks, eye_points, width, height):
    coords = []

    for index in eye_points:
        x = landmarks[index].x * width
        y = landmarks[index].y * height
        coords.append((x, y))

    p1, p2, p3, p4, p5, p6 = coords
    vertical1 = euclidean(p2, p6)
    vertical2 = euclidean(p3, p5)
    horizontal = euclidean(p1, p4)

    if horizontal == 0:
        return 0.3

    return (vertical1 + vertical2) / (2.0 * horizontal)


# ============================================================
# 7. BLINK MORSE DECODER
# ============================================================

class BlinkMorseDecoder:
    def __init__(self):
        self.EAR_THRESHOLD = 0.21
        self.eye_closed = False
        self.closed_start = None
        self.open_start = time.time()
        self.current_symbol = ""
        self.decoded_text = ""
        self.letter_flushed = False
        self.word_space_added = False

    def flush_letter(self):
        if self.current_symbol:
            char = MORSE_TO_CHAR.get(self.current_symbol, '?')
            self.decoded_text += char
            print("Morse:", self.current_symbol, "->", char)
            self.current_symbol = ""

    def backspace(self):
        """Delete one pending Morse mark, or the last decoded character."""
        if self.current_symbol:
            self.current_symbol = self.current_symbol[:-1]
            print("Deleted last Morse symbol. Current Morse:", self.current_symbol)
        elif self.decoded_text:
            self.decoded_text = self.decoded_text[:-1]
            print("Deleted last decoded character. Text:", self.decoded_text)
        else:
            print("Nothing to delete.")

    def add_space(self):
        """Optional keyboard alternative for adding a word space."""
        self.flush_letter()
        if self.decoded_text and not self.decoded_text.endswith(" "):
            self.decoded_text += " "
            print("Space added. Text:", self.decoded_text)
        else:
            print("Space not added (text is empty or already ends with a space).")
        self.word_space_added = True

    def update(self, ear):
        now = time.time()
        closed_now = ear < self.EAR_THRESHOLD

        # Eyes have just closed: classify the blink that just ended
        if closed_now and not self.eye_closed:
            self.eye_closed = True
            self.closed_start = now
            self.letter_flushed = False
            self.word_space_added = False

        # Eyes have just opened: classify the duration of the closure
        elif not closed_now and self.eye_closed:
            self.eye_closed = False
            self.open_start = now

            if self.closed_start is not None:
                closed_duration = now - self.closed_start

                if closed_duration < 0.5:
                    print("Ignored blink:", round(closed_duration, 2), "seconds")
                elif closed_duration < 3:
                    self.current_symbol += "."
                    print("DOT - eyes closed for", round(closed_duration, 2), "seconds")
                else:
                    self.current_symbol += "-"
                    print("DASH - eyes closed for", round(closed_duration, 2), "seconds")

                self.closed_start = None

        # While eyes remain open, complete a letter after 3 seconds,
        # then add one word space after 6 seconds.
        if not self.eye_closed:
            open_duration = now - self.open_start

            if open_duration >= 3 and self.current_symbol and not self.letter_flushed:
                self.flush_letter()
                self.letter_flushed = True

            if open_duration >= 6 and not self.word_space_added:
                # Ensure any pending letter is completed before the space.
                self.flush_letter()
                if self.decoded_text and not self.decoded_text.endswith(" "):
                    self.decoded_text += " "
                    print("WORD SPACE added")
                self.word_space_added = True

    def reset(self):
        self.current_symbol = ""
        self.decoded_text = ""
        self.eye_closed = False
        self.closed_start = None
        self.open_start = time.time()
        self.letter_flushed = False
        self.word_space_added = False
        print("Decoder reset.")


# ============================================================
# 8. CREATE FACE LANDMARKER
# ============================================================

def create_face_landmarker():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return vision.FaceLandmarker.create_from_options(options)


# ============================================================
# 9. MAIN PROGRAM
# ============================================================

def main(video_source=0):
    download_model()

    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print("ERROR: Could not open camera.")
        return

    decoder = BlinkMorseDecoder()
    face_landmarker = create_face_landmarker()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("ERROR: Could not read camera frame.")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp = next_timestamp()
            result = face_landmarker.detect_for_video(mp_image, timestamp)

            # Default to open eyes if no face is detected.
            ear = 0.30

            if result.face_landmarks:
                landmarks = result.face_landmarks[0]
                left_ear = eye_aspect_ratio(landmarks, LEFT_EYE, width, height)
                right_ear = eye_aspect_ratio(landmarks, RIGHT_EYE, width, height)
                ear = (left_ear + right_ear) / 2.0

                for index in LEFT_EYE + RIGHT_EYE:
                    x = int(landmarks[index].x * width)
                    y = int(landmarks[index].y * height)
                    cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            decoder.update(ear)
            status = "CLOSED" if ear < decoder.EAR_THRESHOLD else "OPEN"

            cv2.putText(frame, f"EAR: {ear:.2f}  Eye: {status}",
                        (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.putText(frame, "DOT: close eyes 0.5-3 sec",
                        (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
            cv2.putText(frame, "DASH: close eyes 3+ sec",
                        (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
            cv2.putText(frame, "Open 3 sec: letter | Open 6 sec: space",
                        (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.putText(frame, f"Morse: {decoder.current_symbol}",
                        (20, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

            # Show the decoded message, including spaces, across multiple lines if needed.
            display_text = decoder.decoded_text if decoder.decoded_text else ""
            cv2.putText(frame, f"Decoded: {display_text[-45:]}",
                        (20, height - 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(frame, "Click camera window: [b] backspace [space] space [r] reset [q] quit",
                        (20, height - 65), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

            cv2.imshow("Blink Morse Decoder", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break
            elif key == ord("r"):
                decoder.reset()
            elif key == ord("b") or key == 8 or key == 127:
                decoder.backspace()
            elif key == 32:
                decoder.add_space()

    finally:
        cap.release()
        face_landmarker.close()
        cv2.destroyAllWindows()

    print()
    print("Final decoded text:", decoder.decoded_text)


# ============================================================
# 10. START
# ============================================================

if __name__ == "__main__":
    main(video_source=0)


## 3. Pre-recorded video decoder

The following cell defines the video-processing implementation without automatically opening a placeholder file. Run it, then call `process_video()` in a separate cell with the path to your own video.

Example:

```python
process_video("my_video.mp4", show=True)
```

To save an annotated video:

```python
process_video("my_video.mp4", show=True, save_path="annotated.mp4")
```


In [ ]:
"""Blink-to-Morse-Code Decoder - Pre-recorded video version
-----------------------------------------------------------
Usage:
    python blink_morse_video.py my_video.mp4
    python blink_morse_video.py my_video.mp4 --show
    python blink_morse_video.py my_video.mp4 --save annotated.mp4
    python blink_morse_video.py my_video.mp4 --threshold 0.19 --mirror

Rules (timed using the VIDEO's own clock, not real time):
    DOT         = eyes closed 0.5 to < 3 seconds
    DASH        = eyes closed 3 seconds or more
    LETTER GAP  = eyes open 3 seconds
    WORD SPACE  = eyes open 6 seconds

While previewing with --show:  q = quit, p = pause/resume
"""

import argparse
import os
import sys
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ============================================================
# 1. MORSE TABLE
# ============================================================

MORSE_TO_CHAR = {
    '.-': 'A', '-...': 'B', '-.-.': 'C', '-..': 'D', '.': 'E',
    '..-.': 'F', '--.': 'G', '....': 'H', '..': 'I', '.---': 'J',
    '-.-': 'K', '.-..': 'L', '--': 'M', '-.': 'N', '---': 'O',
    '.--.': 'P', '--.-': 'Q', '.-.': 'R', '...': 'S', '-': 'T',
    '..-': 'U', '...-': 'V', '.--': 'W', '-..-': 'X', '-.--': 'Y',
    '--..': 'Z',
    '-----': '0', '.----': '1', '..---': '2', '...--': '3',
    '....-': '4', '.....': '5', '-....': '6', '--...': '7',
    '---..': '8', '----.': '9'
}


# ============================================================
# 2. MODEL
# ============================================================

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)
MODEL_PATH = "face_landmarker.task"


def download_model():
    if os.path.exists(MODEL_PATH):
        return
    print("Downloading Face Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.")


def create_face_landmarker():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.FaceLandmarker.create_from_options(options)


# ============================================================
# 3. EYE ASPECT RATIO
# ============================================================

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]


def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def eye_aspect_ratio(landmarks, eye_points, width, height):
    coords = [(landmarks[i].x * width, landmarks[i].y * height) for i in eye_points]
    p1, p2, p3, p4, p5, p6 = coords
    horizontal = euclidean(p1, p4)
    if horizontal == 0:
        return 0.3
    return (euclidean(p2, p6) + euclidean(p3, p5)) / (2.0 * horizontal)


# ============================================================
# 4. DECODER (driven by video time in seconds)
# ============================================================

class BlinkMorseDecoder:
    def __init__(self, ear_threshold=0.21):
        self.EAR_THRESHOLD = ear_threshold
        self.eye_closed = False
        self.closed_start = None
        self.open_start = 0.0
        self.current_symbol = ""
        self.decoded_text = ""
        self.letter_flushed = False
        self.word_space_added = False
        self.log = []          # (video_time, event description)

    def _log(self, t, msg):
        self.log.append((t, msg))
        print(f"[{t:7.2f}s] {msg}")

    def flush_letter(self, t):
        if self.current_symbol:
            char = MORSE_TO_CHAR.get(self.current_symbol, '?')
            self.decoded_text += char
            self._log(t, f"LETTER  {self.current_symbol} -> {char}")
            self.current_symbol = ""

    def _classify_closure(self, t):
        if self.closed_start is None:
            return
        dur = t - self.closed_start
        if dur < 0.5:
            self._log(t, f"ignored blink ({dur:.2f}s)")
        elif dur < 3:
            self.current_symbol += "."
            self._log(t, f"DOT   (closed {dur:.2f}s)")
        else:
            self.current_symbol += "-"
            self._log(t, f"DASH  (closed {dur:.2f}s)")
        self.closed_start = None

    def update(self, ear, t):
        closed_now = ear < self.EAR_THRESHOLD

        if closed_now and not self.eye_closed:
            self.eye_closed = True
            self.closed_start = t
            self.letter_flushed = False
            self.word_space_added = False

        elif not closed_now and self.eye_closed:
            self.eye_closed = False
            self.open_start = t
            self._classify_closure(t)

        if not self.eye_closed:
            open_dur = t - self.open_start

            if open_dur >= 3 and self.current_symbol and not self.letter_flushed:
                self.flush_letter(t)
                self.letter_flushed = True

            if open_dur >= 6 and not self.word_space_added:
                self.flush_letter(t)
                if self.decoded_text and not self.decoded_text.endswith(" "):
                    self.decoded_text += " "
                    self._log(t, "WORD SPACE")
                self.word_space_added = True

    def finish(self, t):
        """Call when the video ends so nothing pending is lost."""
        if self.eye_closed:
            self._classify_closure(t)
            self.eye_closed = False
        self.flush_letter(t)


# ============================================================
# 5. MAIN
# ============================================================

def process_video(path, show=False, save_path=None, threshold=0.21, mirror=False):
    if not os.path.exists(path):
        print(f"ERROR: Video file not found: {path}")
        return None

    download_model()

    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print("ERROR: Could not open video.")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or np.isnan(fps):
        fps = 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video: {w}x{h} @ {fps:.2f} fps, ~{total} frames "
          f"(~{total / fps:.1f}s)")

    writer = None
    if save_path:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(save_path, fourcc, fps, (w, h))

    decoder = BlinkMorseDecoder(ear_threshold=threshold)
    landmarker = create_face_landmarker()

    frame_idx = 0
    last_ts_ms = -1
    video_time = 0.0
    paused = False

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            video_time = frame_idx / fps
            ts_ms = int(video_time * 1000)
            if ts_ms <= last_ts_ms:           # MediaPipe needs strictly increasing
                ts_ms = last_ts_ms + 1
            last_ts_ms = ts_ms
            frame_idx += 1

            if mirror:
                frame = cv2.flip(frame, 1)

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(mp_image, ts_ms)

            ear = 0.30  # treat "no face" as eyes open
            face_found = bool(result.face_landmarks)
            if face_found:
                lm = result.face_landmarks[0]
                ear = (eye_aspect_ratio(lm, LEFT_EYE, w, h) +
                       eye_aspect_ratio(lm, RIGHT_EYE, w, h)) / 2.0
                if show or writer:
                    for i in LEFT_EYE + RIGHT_EYE:
                        cv2.circle(frame, (int(lm[i].x * w), int(lm[i].y * h)),
                                   2, (0, 255, 0), -1)

            decoder.update(ear, video_time)

            if show or writer:
                status = "CLOSED" if ear < decoder.EAR_THRESHOLD else "OPEN"
                face_txt = "" if face_found else "  (no face)"
                cv2.putText(frame, f"t={video_time:6.2f}s  EAR: {ear:.2f}  {status}{face_txt}",
                            (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)
                cv2.putText(frame, f"Morse: {decoder.current_symbol}",
                            (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
                cv2.putText(frame, f"Decoded: {decoder.decoded_text[-40:]}",
                            (20, h - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            if writer:
                writer.write(frame)

            if show:
                cv2.imshow("Blink Morse Decoder (video)", frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    break
                if key == ord("p"):
                    paused = True
                while paused:
                    k = cv2.waitKey(50) & 0xFF
                    if k in (ord("p"), ord("q")):
                        paused = False
                        if k == ord("q"):
                            frame_idx = -1
                if frame_idx == -1:
                    break

            if frame_idx % 300 == 0 and total:
                print(f"  ...processed {frame_idx}/{total} frames")

    finally:
        cap.release()
        if writer:
            writer.release()
        landmarker.close()
        if show:
            cv2.destroyAllWindows()

    decoder.finish(video_time)

    print("\n" + "=" * 50)
    print("Final decoded text:", decoder.decoded_text.strip())
    print("=" * 50)

    out_txt = os.path.splitext(path)[0] + "_decoded.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(decoder.decoded_text.strip() + "\n\n--- event log ---\n")
        for t, msg in decoder.log:
            f.write(f"[{t:7.2f}s] {msg}\n")
    print(f"Saved text + event log to: {out_txt}")
    if save_path:
        print(f"Saved annotated video to: {save_path}")

    return decoder.decoded_text


# ---- Jupyter / Spyder / IDLE users: set your video path here ----
VIDEO_PATH = "my_video.mp4"

if __name__ == "__main__" and "ipykernel" not in sys.modules:
    parser = argparse.ArgumentParser(
        description="Decode Morse code from eye blinks in a video file."
    )
    parser.add_argument("video", help="Path to the prerecorded video")
    parser.add_argument("--show", action="store_true", help="Preview the video")
    parser.add_argument("--save", metavar="OUT.mp4", help="Save an annotated video")
    parser.add_argument(
        "--threshold", type=float, default=0.21,
        help="EAR threshold for closed eyes (default: 0.21)"
    )
    parser.add_argument("--mirror", action="store_true", help="Flip frames horizontally")
    args = parser.parse_args()
    process_video(
        args.video, show=args.show, save_path=args.save,
        threshold=args.threshold, mirror=args.mirror
    )


## 4. Timing rules

The default interpretation uses these thresholds:

| Gesture | Interpretation |
|---|---|
| Eyes closed for 0.5 to less than 3 seconds | Dot (`.`) |
| Eyes closed for 3 seconds or longer | Dash (`-`) |
| Eyes open for 3 seconds | Letter boundary |
| Eyes open for 6 seconds | Word boundary |

These timing rules are configurable in the implementation and may need adjustment for reliable use.
